In [7]:
import torch
from torch.utils import data
import re
from torch import nn
from torch.nn import functional as F
import math
import random
batch_size=32
num_step=35
lr=1
epoch=500

In [8]:
def corpus_vocab_maker():
    with open('../data/timemachine.txt','r') as file:
        text=file.read().lower()
    text=re.sub('[^A-Za-z]+',' ',text).strip()
    tokens=list(text)
    idx_to_token=[*(['<unk>']+list(set(tokens)))]
    token_to_idx={
        token:idx for idx,token in enumerate(idx_to_token)
    }
    vocab={
        "idx_to_token":idx_to_token,
        "token_to_idx":token_to_idx
    }
    corpus=torch.tensor([token_to_idx[token] for token in tokens])
    return corpus,vocab

In [9]:
class SequentialTextLoader:
    def __init__(
        self,
        corpus,
        batch_size,
        num_step
    ):
        self.corpus = (
            corpus
            .flatten()
            .long()
        )

        self.batch_size = batch_size
        self.num_step = num_step

    def __iter__(self):
        # 每次进入一个新epoch时，
        # 都重新产生随机切片偏移
        t = random.randint(
            0,
            self.num_step - 1
        )

        # 减1是因为还需要给X制作错开一位的Y
        num_tokens = (
            (
                len(self.corpus)
                - t
                - 1
            )
            // self.batch_size
        ) * self.batch_size

        if num_tokens <= 0:
            raise ValueError(
                "corpus长度不足以构造一个batch"
            )

        x_all = self.corpus[
            t:
            t + num_tokens
        ]

        y_all = self.corpus[
            t + 1:
            t + 1 + num_tokens
        ]

        # 每一行都是一条连续的文本序列
        x_all = x_all.reshape(
            self.batch_size,
            -1
        )

        y_all = y_all.reshape(
            self.batch_size,
            -1
        )

        num_batches = (
            x_all.shape[1]
            // self.num_step
        )

        for start in range(
            0,
            num_batches * self.num_step,
            self.num_step
        ):
            x = x_all[
                :,
                start:start + self.num_step
            ]

            y = y_all[
                :,
                start:start + self.num_step
            ]

            yield x, y


def train_iter_maker(
    corpus,
    batch_size,
    num_step
):
    train_iter = SequentialTextLoader(
        corpus,
        batch_size,
        num_step
    )

    return train_iter

In [10]:
corpus,vocab=corpus_vocab_maker()
device=torch.device("cuda:0")
vocab_size=len(vocab['idx_to_token'])


In [11]:
class rnnnet(nn.Module):
    def __init__(self,input_num,hidden_num,num_layer):
        super().__init__()
        self.input_num=input_num
        self.hidden_num=hidden_num
        self.num_layer=num_layer
        self.rnn=nn.RNN(input_size=input_num,hidden_size=hidden_num,num_layers=num_layer)
        self.output=nn.Linear(hidden_num,input_num)
    def forward(self,x,state):
        x=F.one_hot(x.T.long(),num_classes=self.input_num).float()
        rnn_out,state=self.rnn(x,state)
        rnn_out=rnn_out.reshape(-1,self.hidden_num)
        result=self.output(rnn_out)
        return result,state
    def get_new_state(self,batch_size,device):
        return torch.zeros(self.num_layer,batch_size,self.hidden_num,device=device)



In [12]:
net=rnnnet(vocab_size,256,1).to(device)
trainer=torch.optim.SGD(net.parameters(),lr=lr)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(trainer,T_max=epoch,eta_min=1e-4)
loss = nn.CrossEntropyLoss()
for i in range(epoch):
    net.train()
    train_loss=0.0
    train_count=0.0
    state=None
    train_iter=train_iter_maker(corpus,batch_size,num_step)
    for x,y in train_iter:
        x=x.to(device)
        y=y.T.reshape(-1).long().to(device)
        if(state==None):
            state=net.get_new_state(x.shape[0],device)
        else:
            state=state.detach()
        trainer.zero_grad()
        logit,state=net(x,state)
        l=loss(logit,y)
        l.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(),max_norm=1.0) #梯度裁剪 防止梯度过大
        trainer.step()
        train_loss+=l.item()*len(y)
        train_count+=len(y)
    scheduler.step()
    print(f"第{i+1}epoch的loss:{train_loss/train_count},困惑度：{math.exp(train_loss/train_count)}")

第1epoch的loss:2.488283600125994,困惑度：12.04059190390259
第2epoch的loss:2.2047556545827294,困惑度：9.068035564198508
第3epoch的loss:2.1337377885719397,困惑度：8.446378660068339
第4epoch的loss:2.0736403620088257,困惑度：7.953724917443203
第5epoch的loss:2.021460649255034,困惑度：7.5493438277711515
第6epoch的loss:1.9738326343623074,困惑度：7.198211802146054
第7epoch的loss:1.927006952174298,困惑度：6.868920436439196
第8epoch的loss:1.887569973221073,困惑度：6.603302965853328
第9epoch的loss:1.8522878776897083,困惑度：6.37438666936904
第10epoch的loss:1.8216229700422906,困惑度：6.1818833240877415
第11epoch的loss:1.7899980072851305,困惑度：5.989440531124015
第12epoch的loss:1.7591509826771625,困惑度：5.80750462866492
第13epoch的loss:1.7316086307748573,困惑度：5.649734939950881
第14epoch的loss:1.7103087120241933,困惑度：5.530668598004184
第15epoch的loss:1.68334567856479,困惑度：5.383537460090764
第16epoch的loss:1.6654446705595238,困惑度：5.288024155692785
第17epoch的loss:1.645285432214861,困惑度：5.182488945153996
第18epoch的loss:1.6312017479500214,困惑度：5.110011976496861
第19epoch的loss:1.6102916366

In [13]:
def predict_text(text,net,vocab,num,device):
    output_idx=[vocab['token_to_idx'][text[0]]]
    state=net.get_new_state(1,device)
    net.eval()
    def get_input():
        return torch.tensor(output_idx[-1],dtype=torch.long,device=device).reshape(1,-1)
    with torch.no_grad():
        for token in text[1:]:
            _,state=net(get_input(),state)
            output_idx.append(vocab['token_to_idx'][token])
        for i in range(num):
            y,state=net(get_input(),state)
            y=y[-1].argmax(dim=-1).item()
            output_idx.append(y)
        return "".join([
                vocab['idx_to_token'][index]
                for index in output_idx])

In [14]:
predict_text("world",net,vocab,100,device)

'world me the thing was so hum no shelt s humantnelations as the day and the story i saw the proper a sugg'